<a href="https://colab.research.google.com/github/fcofdezmx/Black_Belt_Project/blob/main/HCO_RAG_Multiple_docs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HCO assitant with RAG**



## **Problem Definition**

HCO is a Cisco Crosswork solutio.

## **How HCO assitant can help**

We will use a **RAG** model to answer questions from HCO documents.







# **Setup**

In [1]:
# We will use the following lines to filter warnings
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Installation for GPU llama-cpp-python
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28  --force-reinstall --upgrade --no-cache-dir -q 2>/dev/null

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 189.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 139.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 121.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 141.0 MB/s eta 0:00:00


In [3]:
# For installing the libraries & downloading models from HF Hub
!pip install -q tiktoken==0.6.0 \
                pypdf==4.0.1 \
                langchain==0.1.1 \
                langchain-community==0.0.13 \
                chromadb==0.4.22 \
                sentence-transformers==2.3.1 \
                huggingface_hub==0.23.2 \
                numpy==1.25.2 2>/dev/null

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.0/284.0 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.4/802.4 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.8 MB/s e

In [4]:
import json
import tiktoken

import pandas as pd

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings
)
from langchain_community.vectorstores import Chroma

from google.colab import userdata, drive

## **LLAMA-CPP**

Add your `HF_TOKEN` in the Secrets section on the left-hand side above the Files menu.
- In the name field enter `HF_TOKEN`.
- In the value field, enter your `access_token`.


In [5]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

In [6]:
model_name_or_path = "TheBloke/Llama-2-13B-chat-GGUF"
model_basename = "llama-2-13b-chat.Q5_K_M.gguf" # the model is in gguf format

In [7]:
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
    )


llama-2-13b-chat.Q5_K_M.gguf:   0%|          | 0.00/9.23G [00:00<?, ?B/s]

In [8]:
lcpp_llm = Llama(
        model_path=model_path,
        n_threads=2,  # CPU cores
        n_batch=512,  # Should be between 1 and n_ctx, consider the amount of VRAM in your GPU.
        n_gpu_layers=43,  # Change this value based on your model and your GPU VRAM pool.
        n_ctx=4096,  # Context window
    )

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


In [9]:
response = lcpp_llm("Tell me about yourself.", max_tokens=500)
response_text = response["choices"][0]["text"]
print(response_text)



I'm just an AI, I don't have a personal life or experiences. However, I can tell you about my capabilities and what I can do to help you with your questions or tasks.

I am trained on a large dataset of text from the internet and can generate human-like responses to a wide range of questions and prompts. I can be used for a variety of tasks such as answering customer queries, providing information on a particular topic, or simply engaging in conversation.

Some examples of what I can do include:

* Answering questions on a variety of topics such as history, science, technology, and more.
* Generating text based on a given prompt or topic.
* Providing information on a particular subject or topic.
* Engaging in conversation and responding to questions in a human-like manner.

I am constantly learning and improving my capabilities, so if there is something specific you would like me to do or learn, please let me know!


In [10]:
response = lcpp_llm("What is a flower?", max_tokens=500)
response_text = response["choices"][0]["text"]
print(response_text)

Llama.generate: prefix-match hit



A flower is a part of a plant that produces seeds, and it's one of the most beautiful and fascinating parts. Flowers are made up of petals, which are the colorful parts that you can see, and they contain the reproductive organs of the plant, like the stigma, style, and stamen.

Flowers come in all shapes, sizes, and colors, and they have different scents and textures. Some flowers are big and showy, while others are small and delicate. Some have vibrant colors, while others are more subtle.

Flowers play an important role in the life cycle of plants. They produce nectar, which attracts pollinators like bees, butterflies, and hummingbirds. These pollinators help transfer pollen from one flower to another, allowing the plants to reproduce.

In addition to their role in plant reproduction, flowers have many other uses. They are often used in decorations, arrangements, and bouquets for special occasions. They are also used in herbal remedies, perfumes, and cosmetics. Some flowers are even

In [42]:
response = lcpp_llm("How to unassign device in Cisco Crosswork Hierarchical Controller version 10 ?", max_tokens=500)
response_text = response["choices"][0]["text"]
print(response_text)

Llama.generate: prefix-match hit




How to unassignment a device from a particular tenant in Cisco Crosswork Hierarchical controller version 10?

Answer:

To unassign a device from a specific tenant in Cisco Crosswork Hierarchical Controller version 10, follow these steps:

Step 1: Log in to the Crosswork Hierarchical Controller with valid credentials.

Step 2: Navigate to the "Tenants" page by clicking on the "Tenants" tab on the top menu bar.

Step 3: Select the tenant for which you want to unassign the device by clicking on the tenant name from the list of available tenants.

Step 4: Once the tenant is selected, navigate to the "Devices" page by clicking on the "Devices" tab on the top menu bar.

Step 5: Select the device you want to unassign from the list of devices assigned to the selected tenant.

Step 6: Click on the "Actions" dropdown menu associated with the device and select "Unassign Device".

Step 7: Confirm the action by clicking on the "Unassign Device" button in the pop-up window.

Step 8: Once the devic

# **Implementing RAG**

## **1 - Loading the PDF, Chunking**

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# pdf_file = "/content/Cisco_Crosswork_Hierarchical_Controller_Administration_Guide.pdf"

In [ ]:
# pdf_loader = PyPDFLoader(pdf_file);

In [12]:
pdf_files = "/content/drive/MyDrive/AI_Projects/Black_Belt_Project/Version 10"

In [14]:
pdf_loader = PyPDFDirectoryLoader(pdf_files);

Here we have used the `PyPDFLoader` because we are working with a single document. Suppose we were dealing with multiple documents in various files within a folder. In that case, we would use the `PyPDFDirectorLoader` to point to this folder. It would then load each file, break it into chunks, and store these chunks in a list. This process involves looping over each file in the directory, chunking the file, and storing the chunks.


In [15]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

In [16]:
microsoft_chunks = pdf_loader.load_and_split(text_splitter)

(Note: Expect that the above cell will take time to execute).

In [17]:
len(microsoft_chunks)

919

## **2 - Vector Store - ChromaDB, Embeddings**

In [18]:
HCO_DOCS = 'hco_docs'

In [19]:
import numpy as np
np.__version__

'1.26.4'

In [20]:
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.9k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [21]:
vectorstore = Chroma.from_documents(
    microsoft_chunks,
    embedding_model,
    collection_name=HCO_DOCS
)

In [22]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

## **3 - RAG Q&A**

### **Prompt Design**

In [23]:
qna_system_message = """
You are an assistant whose work is to review the HCO guide and provide the appropriate answers from the context.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Do not mention anything about the context in your final answer.

If the answer is not found in the context, respond "I don't know".
"""

In [24]:
qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

### **Retrieving the Relevant Documents**

In [43]:
user_input = "How to unassign device in Cisco Crosswork Hierarchical Controller version 10 ?"

In [44]:
relevant_document_chunks = retriever.get_relevant_documents(user_input)

In [45]:
len(relevant_document_chunks)

5

In [46]:
for document in relevant_document_chunks:
    print(document.page_content.replace("\t", " "))
    break

Cisco Crosswork Hierarchical Controller 10.0 Administration Guide  
© 2024 Cisco and/or its affiliates . All rights reserved.  Page 71 of 114  
9. To unassign the device from an adapter, click Unassign device from the adapter . 
10. To assign the device to an adapter, click Assign device to a new adapter . 
 
11. Select the adapter to assign the device to and click Assign .


### **Defining the RAG function for response**




In [47]:
def RAG(user_input):
    """
    Args:
    user_input: Takes a user input for which the response should be retrieved from the vectorDB.
    Returns:
    relevant context as per user query.
    """
    relevant_document_chunks = retriever.get_relevant_documents(user_input)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)



    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""


    # Quering an LLM
    try:
        response = lcpp_llm(
                prompt=prompt,
                max_tokens=500,
                temperature=0,
                top_p=0.95,
                repeat_penalty=1.2,
                top_k=50,
                stop=['INST'],
                echo=False
                )

        prediction =  response["choices"][0]["text"]

    except Exception as e:
        prediction = f'Sorry, I encountered the following error: \n {e}'

    return  prediction

In [48]:
print(RAG("How to unassign device in Cisco Crosswork Hierarchical Controller version 10 ?"))

Llama.generate: prefix-match hit


  Sure, I'd be happy to help! Based on the context you provided, here is the answer to your question:

To unassign a device from an adapter in Cisco Crosswork Hierarchical Controller version 10, follow these steps:

1. In the applications bar, select Services > Device Manager.
2. Select the required adapter.
3. Select the Managed Devices tab.
4. Click on the required device row (not on the link in the Name column).
5. Select the Adapters tab.
6. Click Unassign device from this adapter.
7. Click Save.

That's it! The device will be unassigned from the adapter, but it will remain in the model.


In [40]:
print(RAG("How to assign a device to adapter can be added in Cisco Crossworks HCO version 10 ?"))

Llama.generate: prefix-match hit


  Sure, I'd be happy to help! Based on the context you provided, here is the answer to your question:

To assign a device to an adapter in Cisco Crosswork Hierarchical Controller (HCO) version 10, follow these steps:

1. In the applications bar, select Services > Device Manager.
2. Select the Managed Devices tab.
3. Click on the required device row (not on the link in the Name column).
4. Select the Adapters tab.
5. Click Assign device to a new adapter.
6. Select an adapter and click Assign.
7. Complete the details for the adapter, such as host, port, direct connect, authentication, and enabled.
8. Repeat these steps for as many adapters as required.
9. Click Add Device to add a device to the list of managed devices.
10. Follow the remaining steps in the guide to complete the assignment of devices to adapters.

I hope this helps! Let me know if you have any other questions or need further clarification.


In [41]:
print(RAG("Can you explain what are Regions API ?"))

Llama.generate: prefix-match hit


  Sure, I can help with that! Based on the context provided, it seems like Regions API is a feature in Cisco Crosswork Hierarchical Controller that allows for managing regions and overlays. A region is defined as a geographical area where network sites are located, and an overlay is used to group several regions together.

The Regions API provides endpoints for querying the model to return the region definition, getting the sites in one or more regions, adding regions to an overlay, and getting the sites in an overlay. The API supports various geometry types for regions, including Point, LineString, Polygon, MultiPoint, MultiLineString, and MultiPolygon.

The context also mentions that Cisco will usually collaborate with customers to set up the regions in their model, and that the regions can be exported or imported in GeoJSON or Region POJOs format.


## **4 - Evaluation**

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Llama model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [ ]:
groundedness_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
"""

In [ ]:
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluaton criteria and assign a score.
"""

In [ ]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [ ]:
user_input = "How to add and adapter in Cisco Crossworks HCO version 10 ?"

In [ ]:
relevant_document_chunks = retriever.get_relevant_documents(user_input)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

In [ ]:
# Combine user_prompt and system_message to create the prompt
prompt = f"""[INST]{qna_system_message}\n
            {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
            [/INST]"""

response = lcpp_llm(
        prompt=prompt,
        max_tokens=500,
        temperature=0,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False
        )

answer =  response["choices"][0]["text"]

Llama.generate: prefix-match hit


In [ ]:
print(answer)

  Sure! I'd be happy to help you with your question based on the provided context.

To add an adapter in Cisco Crosswork Hierarchical Controller (HCO) version 10, follow these steps:

1. In the applications bar, select Services > Device Manager.
2. Select the Managed Devices tab.
3. Click Add Device.
4. In the General tab, enter the Name of the adapter.
5. In Network Element Site, click to select the network element in Explorer.
6. Select the Adapters tab and click Assign Device to a new adapter.
7. Select an Adapter and click Assign.

That's it! You have successfully added an adapter in Cisco Crosswork HCO version 10.


In [ ]:
# Combine user_prompt and system_message to create the prompt
groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
            {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
            [/INST]"""

response = lcpp_llm(
        prompt=groundedness_prompt,
        max_tokens=500,
        temperature=0,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False
        )

print(response["choices"][0]["text"])

Llama.generate: prefix-match hit


  Sure, I can help you evaluate the answer based on the provided context and metric. Here are the steps to follow:

Step 1: Evaluate if the answer is derived only from the information presented in the context.

The answer provides a step-by-step guide for adding an adapter in Cisco Crosswork Hierarchical Controller (HCO) version 10, which is based on the information provided in the context. Therefore, the metric is followed to a good extent.

Step 2: Evaluate if the answer adheres to the metric considering the question and context as input.

The question asks how to add an adapter in Cisco Crosswork HCO version 10, and the answer provides clear instructions on how to do so. The answer mentions all the necessary steps required to add an adapter, including selecting the Managed Devices tab, clicking Add Device, entering the Name of the adapter, selecting the network element in Explorer, and assigning the device to an adapter. Therefore, the metric is followed completely.

Step 3: Evaluat

In [ ]:
# Combine user_prompt and system_message to create the prompt
relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
            {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
            [/INST]"""

response = lcpp_llm(
        prompt=relevance_prompt,
        max_tokens=500,
        temperature=0,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False
        )

print(response["choices"][0]["text"])

Llama.generate: prefix-match hit


  Sure, I can help you rate the context as per the evaluation criteria based on the given question and answer. Here are the steps to evaluate the context:

Step 1: Identify the main aspects of the question.
The main aspects of the question are:

* Adding an adapter in Cisco Crosswork Hierarchical Controller (HCO) version 10.
* The process involves selecting a network element, assigning the device to an adapter, and configuring the adapter settings.

Step 2: Evaluate how well the answer addresses the main aspects of the question.
The answer provides clear instructions on how to add an adapter in Cisco Crosswork HCO version 10. It covers all the main aspects of the question, including selecting a network element, assigning the device to an adapter, and configuring the adapter settings. The answer is relevant and complete, addressing all the important aspects of the question.

Step 3: Evaluate the extent to which the metric is followed.
The context follows the metric completely as it prov

# **Conclusion**
- We have learned how to create a Retrieval-Augmented Generation (RAG) based application, which can perform Q&A from documents for quicker, more efficient, and accurate information retrieval.